# 🧬 Dark Manifold V12.1 — Extended Training (OPTIMAL)

## Changes from V12

| Parameter | V12 | V12.1 OPTIMAL |
|-----------|-----|---------------|
| Epochs | 200 | **2000** |
| Batch size | 8 | **64** |
| Trajectory steps | 50 | **100** |
| Learning rate | 3e-4 | **2e-3 → 1e-5** (warmup + cosine) |
| Warmup | 0 | **200 epochs (10%)** |
| Hidden dim | 256 | **512** |

## Goals
- Push trajectory correlation even higher
- Test long-term dynamics (100 steps)
- More stable training with larger batches
- Better generalization with LR schedule

In [ ]:
#@title 1️⃣ Setup
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm
from collections import defaultdict, deque
import matplotlib.pyplot as plt
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nUpload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()

# Parse genes
genes = set()
gene_to_rxns = {}
rxn_to_genes = defaultdict(set)

for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)
        rxn_to_genes[j].add(g)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = str(row['Reaction equation'])
    if '-->' in equation:
        left, right = equation.split('-->')
    elif '<==>' in equation:
        left, right = equation.split('<==>')
    else:
        continue
    for i, name in enumerate(met_names):
        name_lower = name.lower()
        if name_lower in left.lower():
            S[i, j] = -1
        if name_lower in right.lower():
            S[i, j] = +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Key indices
def find_met_idx(keyword, exact=False):
    for i, n in enumerate(met_names):
        if exact:
            if keyword.lower() == n.lower():
                return i
        else:
            if keyword.lower() in n.lower() and 'd' + keyword.lower() not in n.lower():
                return i
    return None

atp_idx = find_met_idx('atp', exact=True) or find_met_idx('atp')
adp_idx = find_met_idx('adp', exact=True) or find_met_idx('adp')
gtp_idx = find_met_idx('gtp', exact=True) or find_met_idx('gtp')

print(f"Data: {n_genes} genes, {n_mets} metabolites, {n_rxns} reactions")
print(f"ATP={atp_idx}, ADP={adp_idx}, GTP={gtp_idx}")

In [ ]:
#@title 3️⃣ V12.1 Enhanced Model

class DarkManifoldV12_1(nn.Module):
    """
    V12.1: Enhanced architecture for extended training.
    
    Changes:
    - Larger hidden dimension (512)
    - Deeper gene encoder (3 layers)
    - Residual connections
    - Dropout for regularization
    """
    
    def __init__(self, n_genes, n_mets, n_rxns, S, G, hidden_dim=512, dropout=0.1):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        self.register_buffer('substrate_mask', (torch.tensor(S) < 0).float())
        
        # Gene regulation matrix
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.1)
        
        # Deep gene encoder with residual
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_rxns),
            nn.Softplus(),
        )
        
        # Metabolite encoder for context
        self.met_encoder = nn.Sequential(
            nn.Linear(n_mets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
            nn.Sigmoid(),
        )
        
        # Kinetics
        self.log_Km = nn.Parameter(torch.randn(n_rxns) * 0.5)
        self.log_Vmax_scale = nn.Parameter(torch.zeros(n_rxns))
        
        # Hamiltonian network
        self.hamiltonian = nn.Sequential(
            nn.Linear(n_mets + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    @property
    def Vmax_scale(self):
        return torch.exp(self.log_Vmax_scale).clamp(0.1, 10.0)
    
    def forward(self, gene_expr, met_conc, dt=0.01):
        batch_size = gene_expr.shape[0]
        
        # Gene regulation
        reg_signal = torch.tanh(gene_expr @ self.W_reg.T)
        regulated = gene_expr * (1.0 + 0.5 * reg_signal)
        regulated = regulated.clamp(min=0.001)
        
        # Enzyme activity from genes
        Vmax_base = self.gene_encoder(regulated)
        Vmax = Vmax_base * self.Vmax_scale.unsqueeze(0)
        
        # Metabolite context modulation
        met_context = self.met_encoder(met_conc)
        
        # Gene-reaction coupling
        enzyme_level = regulated @ self.G
        enzyme_level = enzyme_level / (self.G.sum(dim=0).clamp(min=1))
        
        # Michaelis-Menten
        n_subs = self.substrate_mask.sum(dim=0).clamp(min=1)
        sub_conc = (met_conc @ self.substrate_mask) / n_subs + 0.001
        mm = sub_conc / (self.Km.unsqueeze(0) + sub_conc)
        
        # Combined flux with metabolite context
        flux = Vmax * mm * enzyme_level.clamp(0.01, 2.0) * (0.5 + 0.5 * met_context)
        
        # Stoichiometric dynamics
        dM_dt = flux @ self.S.T
        met_next = (met_conc + dt * dM_dt).clamp(min=0.001)
        
        # Hamiltonian (energy conservation)
        H = self.hamiltonian(torch.cat([met_conc, flux], dim=-1))
        
        return {
            'met_next': met_next,
            'flux': flux,
            'hamiltonian': H,
            'dM_dt': dM_dt,
        }
    
    def rollout(self, gene_expr, met_init, n_steps, dt=0.01):
        met_traj = [met_init.clone()]
        flux_traj = []
        H_traj = []
        
        met = met_init.clone()
        for _ in range(n_steps):
            out = self.forward(gene_expr, met, dt)
            met = out['met_next']
            met_traj.append(met)
            flux_traj.append(out['flux'])
            H_traj.append(out['hamiltonian'])
        
        return {
            'met_trajectory': torch.stack(met_traj, dim=1),
            'flux_trajectory': torch.stack(flux_traj, dim=1),
            'H_trajectory': torch.stack(H_traj, dim=1),
        }


# Initialize
model = DarkManifoldV12_1(
    n_genes=n_genes,
    n_mets=n_mets,
    n_rxns=n_rxns,
    S=S,
    G=G,
    hidden_dim=512,
    dropout=0.1,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"DarkManifoldV12.1: {n_params:,} parameters")

In [ ]:
#@title 4️⃣ Enhanced Generator

class EnhancedGenerator:
    """
    V12.1 Generator with more realistic physics.
    """
    
    def __init__(self, S, G, n_genes, n_mets, n_rxns, device, atp_idx, adp_idx):
        self.S = torch.tensor(S, dtype=torch.float32, device=device)
        self.G = torch.tensor(G, dtype=torch.float32, device=device)
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.device = device
        self.atp_idx = atp_idx
        self.adp_idx = adp_idx
        
        self.substrate_mask = (self.S < 0).float()
        
        # Reaction-specific parameters
        self.Km = torch.rand(n_rxns, device=device) * 2.0 + 0.1
        self.Vmax = torch.rand(n_rxns, device=device) * 0.5 + 0.1
    
    def simulate(self, n_steps=50, batch_size=32):
        # Initial gene expression with variation
        gene_expr = torch.rand(batch_size, self.n_genes, device=self.device) * 1.5 + 0.5
        
        # Initial metabolite concentrations
        met_conc = torch.rand(batch_size, self.n_mets, device=self.device) * 0.8 + 0.3
        
        # Set key metabolites
        if self.atp_idx is not None:
            met_conc[:, self.atp_idx] = 3.5 + torch.rand(batch_size, device=self.device) * 1.0
        if self.adp_idx is not None:
            met_conc[:, self.adp_idx] = 0.3 + torch.rand(batch_size, device=self.device) * 0.4
        
        met_traj = [met_conc.clone()]
        
        for step in range(n_steps):
            # Enzyme activity
            enzyme = gene_expr @ self.G
            enzyme = enzyme / (self.G.sum(dim=0).clamp(min=1))
            
            # Michaelis-Menten
            n_subs = self.substrate_mask.sum(dim=0).clamp(min=1)
            sub_conc = (met_conc @ self.substrate_mask) / n_subs + 0.001
            mm = sub_conc / (self.Km.unsqueeze(0) + sub_conc)
            
            # Flux
            flux = self.Vmax.unsqueeze(0) * enzyme.clamp(0.1, 2.0) * mm
            
            # Dynamics
            dM_dt = flux @ self.S.T
            met_conc = (met_conc + 0.01 * dM_dt).clamp(min=0.001)
            
            # Small noise for realism
            met_conc = met_conc + 0.001 * torch.randn_like(met_conc)
            met_conc = met_conc.clamp(min=0.001)
            
            met_traj.append(met_conc.clone())
        
        return {
            'gene_expr': gene_expr,
            'met_trajectory': torch.stack(met_traj, dim=1),
        }


generator = EnhancedGenerator(S, G, n_genes, n_mets, n_rxns, device, atp_idx, adp_idx)
print("✅ Enhanced generator ready")

In [ ]:
#@title 5️⃣ Training Configuration

# V12.1 OPTIMAL Hyperparameters
N_EPOCHS = 2000
BATCH_SIZE = 64          # Larger for stable gradients + GPU utilization
N_STEPS = 100            # Longer trajectories for long-term dynamics
LR_MAX = 2e-3            # Scaled for batch 64 (sqrt scaling rule)
LR_MIN = 1e-5            # Fine-tuning at end
WARMUP_EPOCHS = 200      # 10% of training for smooth start
WEIGHT_DECAY = 1e-4

print(f"V12.1 OPTIMAL Training Configuration:")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Trajectory steps: {N_STEPS}")
print(f"  LR: {LR_MAX} → {LR_MIN} (warmup + cosine)")
print(f"  Warmup: {WARMUP_EPOCHS} epochs (10%)")
print(f"  Weight decay: {WEIGHT_DECAY}")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR_MAX,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
)

# Custom LR scheduler with warmup
def get_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        # Linear warmup
        return LR_MIN + (LR_MAX - LR_MIN) * epoch / WARMUP_EPOCHS
    else:
        # Cosine annealing
        progress = (epoch - WARMUP_EPOCHS) / (N_EPOCHS - WARMUP_EPOCHS)
        return LR_MIN + (LR_MAX - LR_MIN) * 0.5 * (1 + math.cos(math.pi * progress))

print(f"\n✅ Optimizer configured")

In [ ]:
#@title 6️⃣ Training Loop

losses = []
traj_losses = []
H_losses = []
lrs = []

best_loss = float('inf')
best_state = None

print(f"Training V12.1 for {N_EPOCHS} epochs...")
print("="*60)

pbar = tqdm(range(N_EPOCHS), desc="Training")

for epoch in pbar:
    model.train()
    
    # Update learning rate
    lr = get_lr(epoch)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    lrs.append(lr)
    
    # Generate training data
    with torch.no_grad():
        target = generator.simulate(n_steps=N_STEPS, batch_size=BATCH_SIZE)
    
    gene_expr = target['gene_expr']
    met_init = target['met_trajectory'][:, 0]
    true_traj = target['met_trajectory']
    
    # Forward pass
    pred = model.rollout(gene_expr, met_init.clone(), n_steps=N_STEPS)
    
    # Losses
    loss_traj = F.mse_loss(pred['met_trajectory'], true_traj)
    
    # Hamiltonian conservation (energy shouldn't change much)
    H = pred['H_trajectory'].squeeze(-1)
    loss_H = (H[:, 1:] - H[:, :-1]).pow(2).mean()
    
    # Flux smoothness
    flux = pred['flux_trajectory']
    loss_smooth = (flux[:, 1:] - flux[:, :-1]).pow(2).mean()
    
    # Total loss
    total_loss = 10.0 * loss_traj + 0.1 * loss_H + 0.01 * loss_smooth
    
    # Backward
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    # Record
    losses.append(total_loss.item())
    traj_losses.append(loss_traj.item())
    H_losses.append(loss_H.item())
    
    # Save best
    if total_loss.item() < best_loss:
        best_loss = total_loss.item()
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    # Progress
    pbar.set_postfix({
        'loss': f'{total_loss.item():.4f}',
        'traj': f'{loss_traj.item():.5f}',
        'lr': f'{lr:.1e}',
    })
    
    # Periodic logging
    if (epoch + 1) % 200 == 0:
        print(f"\nEpoch {epoch+1}/{N_EPOCHS}:")
        print(f"  Loss: {total_loss.item():.6f}")
        print(f"  Traj MSE: {loss_traj.item():.6f}")
        print(f"  LR: {lr:.2e}")
        print(f"  Best: {best_loss:.6f}")

# Load best model
model.load_state_dict(best_state)

print("\n" + "="*60)
print(f"Training complete!")
print(f"  Final loss: {losses[-1]:.6f}")
print(f"  Best loss: {best_loss:.6f}")
print(f"  Loss reduction: {100*(1 - best_loss/losses[0]):.1f}%")

In [ ]:
#@title 7️⃣ Training Visualization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss curve
axes[0, 0].semilogy(losses, alpha=0.7, label='Total')
axes[0, 0].semilogy(traj_losses, alpha=0.7, label='Trajectory')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (log)')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# LR schedule
axes[0, 1].plot(lrs)
axes[0, 1].axvline(x=WARMUP_EPOCHS, color='r', linestyle='--', label='Warmup end')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Learning Rate')
axes[0, 1].set_title('LR Schedule (Warmup + Cosine)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Zoomed loss (last 500 epochs)
axes[1, 0].plot(losses[-500:], alpha=0.7)
axes[1, 0].set_xlabel('Epoch (last 500)')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Loss (Last 500 Epochs)')
axes[1, 0].grid(True, alpha=0.3)

# Hamiltonian loss
axes[1, 1].semilogy(H_losses, alpha=0.7, color='green')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('H Loss (log)')
axes[1, 1].set_title('Hamiltonian Conservation')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('V12.1 Training Progress (2000 epochs)', fontsize=14)
plt.tight_layout()
plt.savefig('training_v12_1.png', dpi=150)
plt.show()

In [ ]:
#@title 8️⃣ Trajectory Evaluation

model.eval()

# Multiple test samples
n_test = 5
correlations = []
mses = []

for i in range(n_test):
    with torch.no_grad():
        test = generator.simulate(n_steps=N_STEPS, batch_size=1)
        gene_expr = test['gene_expr']
        met_init = test['met_trajectory'][:, 0]
        true_traj = test['met_trajectory'][0].cpu().numpy()
        
        pred = model.rollout(gene_expr, met_init.clone(), n_steps=N_STEPS)
        pred_traj = pred['met_trajectory'][0].cpu().numpy()
    
    corr = np.corrcoef(true_traj.flatten(), pred_traj.flatten())[0, 1]
    mse = np.mean((true_traj - pred_traj)**2)
    correlations.append(corr)
    mses.append(mse)

print(f"Test Results ({n_test} samples):")
print(f"  Correlation: {np.mean(correlations):.4f} ± {np.std(correlations):.4f}")
print(f"  MSE: {np.mean(mses):.6f} ± {np.std(mses):.6f}")

# Plot best test case
with torch.no_grad():
    test = generator.simulate(n_steps=N_STEPS, batch_size=1)
    pred = model.rollout(test['gene_expr'], test['met_trajectory'][:, 0], n_steps=N_STEPS)

true_traj = test['met_trajectory'][0].cpu().numpy()
pred_traj = pred['met_trajectory'][0].cpu().numpy()
final_corr = np.corrcoef(true_traj.flatten(), pred_traj.flatten())[0, 1]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Key metabolites
indices = [atp_idx or 0, adp_idx or 1, 2, 3, 4, 5]
labels = ['ATP', 'ADP', 'Met 2', 'Met 3', 'Met 4', 'Met 5']

for ax, idx, label in zip(axes.flatten(), indices, labels):
    if idx < true_traj.shape[1]:
        ax.plot(true_traj[:, idx], 'b-', lw=2, label='True')
        ax.plot(pred_traj[:, idx], 'r--', lw=2, label='V12.1')
        ax.set_title(f'{label}')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle(f'V12.1 Trajectory Prediction (Corr={final_corr:.4f})', fontsize=14)
plt.tight_layout()
plt.savefig('trajectory_v12_1.png', dpi=150)
plt.show()

In [ ]:
#@title 9️⃣ Essentiality Analysis (Same as V12)

# Reuse V12 essentiality logic
class QuickEssentialityAnalyzer:
    def __init__(self, S, G, genes, met_names, rxn_to_genes):
        self.S = S
        self.G = G
        self.genes = genes
        self.met_names = met_names
        self.rxn_to_genes = rxn_to_genes
        self.n_genes = len(genes)
        
        self.met_produced_by = defaultdict(list)
        self.met_consumed_by = defaultdict(list)
        
        for rxn in range(S.shape[1]):
            for met in range(S.shape[0]):
                if S[met, rxn] > 0:
                    self.met_produced_by[met].append(rxn)
                elif S[met, rxn] < 0:
                    self.met_consumed_by[met].append(rxn)
    
    def analyze_gene(self, gene_idx):
        rxns = np.where(self.G[gene_idx] > 0)[0]
        n_rxns = len(rxns)
        
        # Dead-end check
        is_dead_end = False
        for rxn in rxns:
            for met in range(self.S.shape[0]):
                if self.S[met, rxn] < 0:
                    other_consumers = [r for r in self.met_consumed_by[met] if r != rxn]
                    if len(other_consumers) == 0 and len(self.met_produced_by[met]) > 0:
                        is_dead_end = True
                        break
        
        is_essential = n_rxns >= 2 or is_dead_end
        
        return {
            'gene': self.genes[gene_idx],
            'n_reactions': n_rxns,
            'dead_end': is_dead_end,
            'is_essential': is_essential,
        }
    
    def analyze_all(self):
        return [self.analyze_gene(i) for i in range(self.n_genes)]


analyzer = QuickEssentialityAnalyzer(S, G, genes, met_names, rxn_to_genes)
ess_results = analyzer.analyze_all()

n_essential = sum(1 for r in ess_results if r['is_essential'])
n_dead_end = sum(1 for r in ess_results if r['dead_end'])
n_hub = sum(1 for r in ess_results if r['n_reactions'] >= 2)

print(f"Essentiality Results:")
print(f"  Essential: {n_essential}/{n_genes} ({100*n_essential/n_genes:.1f}%)")
print(f"  Dead-end: {n_dead_end}")
print(f"  Hub (≥2 rxns): {n_hub}")

In [ ]:
#@title 🔟 Save V12.1

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'traj_losses': traj_losses,
    'H_losses': H_losses,
    'lrs': lrs,
    'version': 'v12.1_extended_training',
    
    'training_config': {
        'epochs': N_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr_max': LR_MAX,
        'lr_min': LR_MIN,
        'warmup': WARMUP_EPOCHS,
        'weight_decay': WEIGHT_DECAY,
        'hidden_dim': 512,
    },
    
    'essentiality': {
        'results': ess_results,
    },
    
    'metrics': {
        'trajectory_corr': float(np.mean(correlations)),
        'trajectory_corr_std': float(np.std(correlations)),
        'trajectory_mse': float(np.mean(mses)),
        'best_loss': float(best_loss),
        'final_loss': float(losses[-1]),
        'n_essential': n_essential,
        'pct_essential': 100 * n_essential / n_genes,
    },
}

torch.save(save_dict, 'dark_manifold_v12_1.pt')

print("="*70)
print("V12.1 FINAL SUMMARY")
print("="*70)
print(f"""
TRAINING:
  Epochs: {N_EPOCHS}
  Batch size: {BATCH_SIZE}
  LR: {LR_MAX} → {LR_MIN}
  Best loss: {best_loss:.6f}
  
TRAJECTORY PREDICTION:
  Correlation: {np.mean(correlations):.4f} ± {np.std(correlations):.4f}
  MSE: {np.mean(mses):.6f}
  
ESSENTIALITY:
  Essential: {n_essential}/{n_genes} ({100*n_essential/n_genes:.1f}%)

COMPARISON:
  V12:   200 epochs, Corr=1.000, 50.3% essential
  V12.1: 2000 epochs, Corr={np.mean(correlations):.4f}, {100*n_essential/n_genes:.1f}% essential
""")

from google.colab import files
files.download('dark_manifold_v12_1.pt')
files.download('training_v12_1.png')
files.download('trajectory_v12_1.png')